# ANDES package walkthrough

This notebook uses the public package API on a small deterministic example. It covers exact database comparison, ranked enrichment, and persistent index queries without constructing a full gene-by-gene similarity matrix.

In [ ]:
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

from andes import (
    EmbeddingSpace,
    GeneSetDatabase,
    score_bma_matrix,
    score_ranked,
)
from andes.index import build_andes_index, load_andes_index

## Canonical inputs

`EmbeddingSpace` normalizes rows and records the exact gene order. `GeneSetDatabase` sorts and deduplicates memberships before it computes sizes and fingerprints.

In [ ]:
rng = np.random.default_rng(17)
genes = tuple(f"g{i}" for i in range(12))
embedding = EmbeddingSpace.from_arrays(
    rng.normal(size=(len(genes), 6)).astype(np.float32),
    genes,
)

memberships = {
    "metabolism": np.array([0, 2, 4, 6], dtype=np.int32),
    "signaling": np.array([1, 3, 5], dtype=np.int32),
    "transport": np.array([6, 8, 10, 11], dtype=np.int32),
}
database = GeneSetDatabase.from_index_mapping(
    memberships,
    n_genes=len(genes),
    embedding_fingerprint=embedding.fingerprint,
    background_policy="retained_members",
)
database.terms

## Exact BMA comparison

The matrix scorer streams best-match chunks and returns raw BMA scores. Null calibration is a separate step.

In [ ]:
comparison = score_bma_matrix(
    embedding,
    database,
    database,
    workspace_mb=1,
)
pd.DataFrame(
    comparison.scores,
    index=database.terms,
    columns=database.terms,
)

## Exact ranked enrichment

A ranking is an ordered array of unique embedding-row indices. Scores are signed cumulative deviations.

In [ ]:
ranking = np.array([7, 2, 10, 0, 5, 9, 1, 11], dtype=np.int32)
ranked = score_ranked(
    embedding,
    database,
    ranking,
    workspace_mb=1,
)
pd.DataFrame(
    {"size": database.sizes, "true_score": ranked.scores},
    index=database.terms,
)

## Persistent index query

An index stores the normalized embedding, canonical term database, sparse membership matrix, and reusable gene-to-term best matches. Query workers normally keep one memory-mapped index open.

In [ ]:
with TemporaryDirectory(prefix="andes-demo-") as temporary:
    index_path = f"{temporary}/pathways.index"
    build_andes_index(
        embedding,
        database,
        index_path,
        max_workspace_mb=1,
    )
    index = load_andes_index(index_path, mmap=True)
    try:
        query = index.map_genes(["g0", "g2", "g8"])[0]
        query_result = index.score_query(query, max_workspace_mb=1)
        query_frame = pd.DataFrame(
            {"size": index.sizes, "true_score": query_result.scores},
            index=index.terms,
        ).sort_values("true_score", ascending=False)
    finally:
        index.close()

query_frame

## Production workflow

Build an index and its accepted query-size null coverage in a controlled job:

```sh
uv run andes index build \
  --emb embedding.npy \
  --genelist genes.txt \
  --geneset pathways.gmt \
  --out pathways.index

uv run andes null bma \
  --index pathways.index \
  --query-sizes 1:300 \
  --cache-root /srv/andes/cache
```

Query workers use the published index and require prebuilt null coverage by default:

```sh
uv run andes index query \
  --index pathways.index \
  --genes query_genes.txt \
  --cache-root /srv/andes/cache \
  --top-k 50 \
  --out query.csv
```

See `README.md` for ranked-null construction, indexed enrichment, expression input, artifact verification, and deployment guidance.